# Step 4 of the PW: Future value of the firms (and the data of the Step 5 Magic Quadrant)

| Question to answer (Presentation deck, Step 4) | How we measure it in this notebook |
|---|---|
| What does "future value" mean for Electrolux? | **Target:** a family in the top 10% of 5-year forward citations of its cohort: priority year × main CPC subclass × office of first filing, coarser when the cohort is small (breakthrough impact, Fleming 2001; the office removes the national habits of citation) (Section 2) |
| Which universe trains and tests the model? | **Robotics universe** (`B25J` + `B62D57`, stratified 2008–2025, ~30k families) containing all humanoid families of Step 0; the extension families of Step 3, Section 11 are scored but never used to train (Section 1) |
| Which signals exist at filing, with no leakage? | **Point-in-time features**: only what is known 18 months after the first priority: offices, codes, inventors and applicants of the applications filed within 12 months; backward citations and examiner X/Y/I categories of the publications published within 18 months (TLS212, TLS215); recombination, momentum, originality; text and science signals for the sector (Section 3) |
| Does the model beat the literature baselines out of time? | **Rolling out-of-time backtest**: the model is **chosen on 2014–2015** and **reported on 2016–2017**; PR-AUC, Precision/Recall/Lift@10%; the gain over the classic benchmark with a **paired bootstrap** on the same families (Section 4) |
| Which signals drive the predictions? | Standardized logistic coefficients and permutation importance (Section 5) |
| Are the probabilities usable at firm level? | **Isotonic calibration** on a held-out year, reliability table; **distribution shift** between training and scored families (PSI, adversarial validation); **shrinkage** of small firms towards the sector prior (Section 6) |
| Does the firm score predict what firms did next? | **Firm-level backtest**: score the firms as of 2016 and compare with their later activity and their realised breakthroughs, against size and PCT baselines (Section 6.3) |
| Where do the firms sit on the Step 5 Quadrant? | The **common set of firms** of Step 3, Section 11 gets a future score here (zero when the firm filed nothing recent); present (Step 3, Section 11) and future scores of the same firms go to `outputs/step4/magic_quadrant_data.csv` (Section 7) |

**Contents**

1. Setup and robotics training universe
2. Target: breakthrough impact, office-aware cohorts
3. Point-in-time features
4. Out-of-time backtest: validation years choose, test years report
5. Interpretability
6. Scoring the recent families and the common set of firms (calibration, shift, shrinkage, firm-level backtest)
7. Magic Quadrant data, sanity checks and deck answer


## 1. Setup, configuration & robotics training universe

Following our group decision (`docs/step4_plan_imma.md`) and **Theory 6**, the **training universe** is wider than the 2,907-family humanoid sector (which has only ~480–800 families in complete 5-year citation cohorts), while **firm scoring** stays strictly on the validated humanoid sector (plus a clearly flagged robotics extension for missing players like AgiBot, Unitree, and Boston Dynamics). All outputs (figures, tables, caches, HTML gallery) write into git-ignored `outputs/step4/`.


In [ ]:
import builtins as _builtins
import html as _html
import json
import re
import sys
import textwrap
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.inspection import permutation_importance
from sqlalchemy import and_, func, or_

sys.path.insert(0, str(Path("src").resolve()))
from future_score import (
    aggregate_firm_future_scores,
    bootstrap_metric_ci,
    compute_arts_lexical_novelty,
    compute_causal_recombination_novelty,
    compute_causal_subfield_momentum,
    compute_dynamic_network_at_age,
    compute_originality_and_tct,
    label_top_decile_by_cohort,
    make_classifiers,
    precision_recall_lift_at_k,
    rule_based_future_score,
    run_rolling_oot_backtest,
    adversarial_validation_auc,
    early_rows,
    fit_isotonic_calibrator,
    label_top_decile_hierarchical,
    paired_bootstrap_diff,
    population_stability_index,
    reliability_table,
    spearman_with_ci,
)
from config import load_config
from quadrant import common_future_scores, quadrant_label

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS209_APPLN_IPC,
    TLS211_PAT_PUBLN,
    TLS212_CITATION,
    TLS215_CITN_CATEG,
    TLS224_APPLN_CPC,
    TLS228_DOCDB_FAM_CITN,
)


In [ ]:
# ---------------------------------------------------------------- environment & parameters
ENV = "PROD"
RANDOM_STATE = 34
YEAR_MIN, YEAR_MAX = 2008, 2025          # 2008-2025 covers modern robotics + full burn-in before T=2014
LAST_COMPLETE_YEAR = 2022
HORIZON = 5                              # 5-year forward citation window (complete up to 2017)
LAST_LABEL_YEAR = LAST_COMPLETE_YEAR - HORIZON   # 2017
CFG = load_config()                      # config/sector.yaml: the choices shared with Step 3
FCFG, QCFG = CFG["future_score"], CFG["quadrant"]
VALIDATION_YEARS = FCFG["validation_years"]   # 2014-2015: the model and the feature set are CHOSEN here
TEST_YEARS = FCFG["test_years"]               # 2016-2017: the numbers of the deck are REPORTED here (never used to choose)
BACKTEST_YEARS = sorted(VALIDATION_YEARS + TEST_YEARS)
LOOKBACK = 5                             # train strictly on [T - LOOKBACK, T - 1]
CALIBRATION_YEAR = FCFG["calibration_year"]   # the final model is trained before this year and calibrated on it
FEATURE_AGE_DAYS = int(round(FCFG["feature_age_months"] * 30.44))   # point-in-time: what is known 18 months after priority
SHRINK_K = FCFG["shrinkage_strength"]
FUTURE_WEIGHTS = FCFG["weights"]
APPLY_YEARS = tuple(FCFG["apply_years"])      # recent wave scored for the future score (x-axis)
TOP_SHARE = 0.10                         # top-decile breakthrough target (Fleming 2001)
MIN_COHORT = 30
STRAT_PER_YEAR = 1_600                   # stratified non-sector robotics families per year (~28.8k + 2.9k sector)
CHUNK = 8_000
DUMMY_ID_FROM = 900_000_000

DATA_DIR = next((p for p in (Path("data") / ENV / "humanoid_robots_v2",
                             Path("..") / "tutorial_0" / "data" / ENV / "humanoid_robots_v2")
                 if (p / "dataset_metadata.json").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Run Step 0 on PROD first: data/PROD/humanoid_robots_v2/dataset_metadata.json not found")

WIDE_NET_FILE = Path("wide_net/outputs/wide_net/wide_net_families.csv")
SECTOR_IDS_FILE = Path(f"ids/sector_family_ids_v3_{ENV}.csv")
STEP3_DIR = Path("outputs/step3")
TEXT_NOVELTY_FILE = Path(f"outputs/t2_text/text_novelty_{ENV}_rule_v3.csv")
OPENALEX_FILE = STEP3_DIR / "science_proximity_openalex.csv"

OUT_DIR = Path("outputs/step4")
CACHE_DIR = OUT_DIR / "cache"
FIGURES_DIR = OUT_DIR / "figures"
TABLES_DIR = OUT_DIR / "tables"
for _d in (OUT_DIR, CACHE_DIR, FIGURES_DIR, TABLES_DIR):
    _d.mkdir(parents=True, exist_ok=True)
ANSWERS_FILE = OUT_DIR / "answers.txt"
ANSWERS_FILE.write_text("", encoding="utf-8")

# ---------------------------------------------------------------- auto-export hooks (git-ignored outputs/step4/)
_EXPORT_LOG = {"figures": [], "tables": [], "answers": []}


def _slug(text, default="item"):
    s = re.sub(r"[^a-zA-Z0-9]+", "_", str(text or "")).strip("_").lower()
    return s[:60] if s else default


def _fig_title(fig):
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text().strip():
        return fig._suptitle.get_text().strip()
    for ax in fig.axes:
        t = ax.get_title().strip()
        if t:
            return t
    return "figure"


if not hasattr(plt, "_orig_show_step4"):
    plt._orig_show_step4 = plt.show


def _auto_save_show(*args, **kwargs):
    for fn in plt.get_fignums():
        fig = plt.figure(fn)
        if getattr(fig, "_step4_exported", False):
            continue
        fig._step4_exported = True
        idx = len(_EXPORT_LOG["figures"]) + 1
        title = _fig_title(fig)
        fname = f"{idx:02d}_{_slug(title, 'figure')}.png"
        fig.savefig(FIGURES_DIR / fname, dpi=150, bbox_inches="tight")
        _EXPORT_LOG["figures"].append({"file": f"figures/{fname}", "title": title})
    return plt._orig_show_step4(*args, **kwargs)


plt.show = _auto_save_show

if not hasattr(_builtins, "_orig_display_step4"):
    _base_disp = globals().get("display", getattr(_builtins, "display", None))
    if _base_disp is None:
        try:
            from IPython.display import display as _base_disp
        except Exception:
            _base_disp = print
    _builtins._orig_display_step4 = _base_disp


def display(*objs, **kwargs):
    for o in objs:
        if isinstance(o, (pd.DataFrame, pd.Series)):
            idx = len(_EXPORT_LOG["tables"]) + 1
            fname = f"{idx:02d}_table.csv"
            o.to_csv(TABLES_DIR / fname)
            _EXPORT_LOG["tables"].append({"file": f"tables/{fname}", "rows": int(len(o))})
    return _builtins._orig_display_step4(*objs, **kwargs)


def answer_box(question, headline, detail=""):
    _EXPORT_LOG["answers"].append({"question": question, "headline": headline, "detail": detail})
    with open(ANSWERS_FILE, "a", encoding="utf-8") as fh:
        fh.write(f"ANSWER · {question}\n{headline}\n{detail}\n" + "-" * 80 + "\n")
    print("-" * 80)
    print(f"ANSWER · {question}")
    for txt in (headline, detail):
        if txt:
            print(textwrap.fill(txt, width=80))
    print("-" * 80)


def load_step0(name):
    for suf, fn in ((".parquet", pd.read_parquet), (".pkl", pd.read_pickle)):
        if (DATA_DIR / f"{name}{suf}").exists():
            return fn(DATA_DIR / f"{name}{suf}")
    raise FileNotFoundError(f"{name} not in {DATA_DIR}")


def write_step4_report_index():
    (OUT_DIR / "run_manifest.json").write_text(json.dumps(_EXPORT_LOG, indent=2), encoding="utf-8")
    parts = [
        "<!doctype html><html><head><meta charset='utf-8'><title>Step 4 Future Score Outputs</title>",
        "<style>body{font-family:system-ui,sans-serif;max-width:1100px;margin:2rem auto;padding:0 1rem;color:#222}"
        "img{max-width:100%;border:1px solid #ddd;border-radius:6px;margin:0.5rem 0 1.5rem}"
        ".box{background:#f8f9fa;border-left:4px solid #e67e22;padding:0.8rem 1rem;margin:1rem 0}</style></head><body>",
        "<h1>Step 4 · Future Score &amp; Backtest Gallery</h1>",
    ]
    for a in _EXPORT_LOG["answers"]:
        parts.append(f"<div class='box'><strong>{_html.escape(a['question'])}</strong><br>{_html.escape(a['headline'])}<br><small>{_html.escape(a['detail'])}</small></div>")
    parts.append(f"<h2>Figures ({len(_EXPORT_LOG['figures'])})</h2>")
    for f in _EXPORT_LOG["figures"]:
        parts.append(f"<h3>{_html.escape(f['title'])}</h3><img src='{_html.escape(f['file'])}'>")
    parts.append(f"<h2>Tables ({len(_EXPORT_LOG['tables'])})</h2><ul>")
    for t in _EXPORT_LOG["tables"]:
        parts.append(f"<li><a href='{_html.escape(t['file'])}'>{_html.escape(t['file'])}</a> ({t['rows']} rows)</li>")
    parts.append("</ul></body></html>")
    idx_path = OUT_DIR / "index.html"
    idx_path.write_text("\n".join(parts), encoding="utf-8")
    print(f"Exported {len(_EXPORT_LOG['figures'])} figures, {len(_EXPORT_LOG['tables'])} tables -> {idx_path}")
    return idx_path


print(f"Step 4 configured | ENV={ENV} | labels {YEAR_MIN}-{LAST_LABEL_YEAR} | validation {VALIDATION_YEARS} | "
      f"test {TEST_YEARS} | calibration {CALIBRATION_YEAR} | features at {FCFG['feature_age_months']} months | score {APPLY_YEARS}")


**Our answer for the deck (Steps 4 → 5).** *To write after the PROD rerun* (review of 28 Nov 2026). The previous answer was built on a future score that existed only for the recent filers and a present score that existed only for the old ones, on uncalibrated probabilities and on a comparison chosen on the test years; its numbers are no longer valid. Write it from Section 7: for each quadrant, the firms with their present and future scores, the number of families each rests on, their flags (*provisional*, *unvalidated*), and their technological proximity to Electrolux; and from Section 4, the gain of our model over the classic benchmark **on the test years, with its paired interval**.

**Limits that we declare.**
- **Training sample vs. sector:** models are trained on a stratified robotics universe (`B25J` + `B62D57` + sector); the humanoid-only test slice is small and its intervals are wide.
- **Title-based sector boundary:** the humanoid makers that do not title their patents "humanoid robot" enter through the extension of Step 3, Section 11, flagged until its reading passes.
- **Recent families:** families of 2024–2025 have not all reached the 18 months of the point-in-time features (Section 3 counts them); their probabilities predict early technological impact, not revenue.

### 1.1 Building the robotics training universe & cached PATSTAT queries

We combine **all 2,907 humanoid families** (Step 0, rule v3) with a **stratified robotics universe** (`wide_net_families.csv` if available, otherwise PATSTAT `B25J` manipulators + `B62D57` legged/walking robots, sampled up to 1,600 non-sector families per priority year across `2008–2025` with fixed seed 34). Every PATSTAT query is cached under `outputs/step4/cache/` so reruns take seconds.


In [ ]:
patstat = PatstatClient(env=ENV)
db = patstat.orm()
REAL = (TLS201_APPLN.ipr_type == "PI", TLS201_APPLN.appln_id < DUMMY_ID_FROM)


def in_chunks(ids, make_query, chunk=CHUNK):
    ids = [int(i) for i in pd.unique(pd.Series(list(ids)))]
    parts = [patstat.df(make_query(ids[s:s + chunk])) for s in range(0, len(ids), chunk)]
    parts = [p for p in parts if len(p)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def cached(name, fn, *args):
    p = CACHE_DIR / f"{name}.parquet"
    if p.exists():
        return pd.read_parquet(p)
    t0 = time.time()
    df = fn(*args)
    df.to_parquet(p, index=False)
    print(f"cached {name}: {len(df):,} rows in {time.time() - t0:.1f}s")
    return df


sector_families = load_step0("families")
sector_ids = set(sector_families["docdb_family_id"].astype(int))
if SECTOR_IDS_FILE.exists():
    sector_ids |= set(pd.read_csv(SECTOR_IDS_FILE)["docdb_family_id"].astype(int))


QUAD_FAMS_FILE = STEP3_DIR / "quadrant_families.csv"      # Step 3, Section 11 (IDs only)


def build_universe():
    if WIDE_NET_FILE.exists():
        wn = pd.read_csv(WIDE_NET_FILE)
        wn = wn[wn["priority_year"].between(YEAR_MIN, YEAR_MAX)].copy()
        wn["in_sector_v3"] = wn["docdb_family_id"].isin(sector_ids)
    else:
        ipc_clean = func.replace(TLS209_APPLN_IPC.ipc_class_symbol, " ", "")
        q = (
            db.query(
                TLS201_APPLN.docdb_family_id.label("docdb_family_id"),
                func.min(TLS201_APPLN.earliest_filing_year).label("priority_year"),
            )
            .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
            .filter(
                *REAL,
                or_(ipc_clean.like("B25J%"), ipc_clean.like("B62D57%")),
                TLS201_APPLN.earliest_filing_year.between(YEAR_MIN, YEAR_MAX),
            )
            .group_by(TLS201_APPLN.docdb_family_id)
        )
        wn = patstat.df(q)
        wn = wn[wn["priority_year"].between(YEAR_MIN, YEAR_MAX)].copy()
        wn["in_sector_v3"] = wn["docdb_family_id"].isin(sector_ids)

    # Keep ALL sector families + stratified sample of non-sector robotics families (<= STRAT_PER_YEAR per year)
    sec_part = sector_families[sector_families["priority_year"].between(YEAR_MIN, YEAR_MAX)][
        ["docdb_family_id", "priority_year"]
    ].drop_duplicates("docdb_family_id").assign(in_sector_v3=True)

    non_sec = wn[~wn["docdb_family_id"].isin(sector_ids)][["docdb_family_id", "priority_year"]].drop_duplicates("docdb_family_id")
    sampled_parts = []
    for yr, grp in non_sec.groupby("priority_year"):
        sampled_parts.append(grp.sample(min(len(grp), STRAT_PER_YEAR), random_state=RANDOM_STATE))
    non_sec_sample = pd.concat(sampled_parts, ignore_index=True).assign(in_sector_v3=False) if sampled_parts else pd.DataFrame(columns=["docdb_family_id", "priority_year", "in_sector_v3"])

    # the extension of Step 3, Section 11 (robotics families of the humanoid makers our title rule misses): scored in
    # Section 6, never used to train or test (they are not a random sample of robotics)
    ext_part = pd.DataFrame(columns=["docdb_family_id", "priority_year", "in_sector_v3", "origin"])
    if QUAD_FAMS_FILE.exists():
        qf = pd.read_csv(QUAD_FAMS_FILE)
        ext_part = (qf[qf["source"] == "extension"][["docdb_family_id", "priority_year"]].drop_duplicates("docdb_family_id")
                    .assign(in_sector_v3=False, origin="extension"))
        ext_part = ext_part[ext_part["priority_year"].between(YEAR_MIN, YEAR_MAX)]
    univ = pd.concat([sec_part.assign(origin="sector"), non_sec_sample.assign(origin="sample"), ext_part],
                     ignore_index=True).drop_duplicates("docdb_family_id")
    univ["docdb_family_id"] = univ["docdb_family_id"].astype(int)
    univ["priority_year"] = univ["priority_year"].astype(int)
    return univ


net = cached("robotics_universe_v2", build_universe).set_index("docdb_family_id")
net["forced_extension"] = net["origin"].eq("extension")
FAMILIES = net.index.tolist()
# every later cache depends on the universe: one folder per universe, so a new universe never reuses old queries
import hashlib
_fp = hashlib.md5(",".join(map(str, sorted(FAMILIES))).encode()).hexdigest()[:10]
CACHE_DIR = OUT_DIR / "cache" / f"universe_{_fp}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Universe: {len(net):,} families ({int(net['in_sector_v3'].sum()):,} humanoid sector | "
      f"{int(net['origin'].eq('sample').sum()):,} robotics controls | {int(net['forced_extension'].sum()):,} extension families, scored only)")
print("Families by priority year (2008-2025):")
display(net.groupby("priority_year").agg(total=("in_sector_v3", "size"), humanoid=("in_sector_v3", "sum")).T)


> **⚠ Rerun needed (review of 28 Nov 2026).** The code above this reading changed; the numbers below are those of the previous PROD run and must be checked, and rewritten if they moved, after *Restart and Run All* on PROD. The universe now also holds the extension families of Step 3, Section 11 (scored, never trained on).

**Our reading of the training universe.** The stratified universe combines **2,721 humanoid families** (`2008–2025`, all rule-v3 families in the window) with **28,800 robotics control families** (`B25J` + `B62D57`, 1,600 per year), for **31,521 families** in total. This solves the small-sample bottleneck of training on humanoids alone (~480–800 families in complete cohorts) while keeping the evaluation anchored on both the full robotics universe and the humanoid sector alone.


## 2. Basic family attributes and the target

We query PATSTAT for applications, CPC/IPC subclasses (per application, so that we know which codes existed at filing) and 5-year forward citations across the universe, then build the target on the complete `2008–2017` cohorts:

- **Target (`valuable`)**: top 10% of 5-year forward citations within the family's cohort **priority year × main subclass × office of first filing** (CN, US, JP, KR, EP, other), falling back to year × subclass and then year when the cohort has fewer than 30 families (Fleming 2001 tail-breakthrough target). Without the office, a family first filed in China competes with US families that cite and are cited by other habits (Step 3, Sections 3.3 and 6.3): the model would learn the office, not the invention.
- The secondary "international" target of the first version (2+ offices or a US/EP grant) is **dropped**: the feature `early_offices_12m` implies it, so its PR-AUC of 0.996 was mechanical, not evidence.

In [ ]:
def fetch_applications(ids):
    return in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.earliest_filing_year,
        TLS201_APPLN.earliest_filing_date,
        TLS201_APPLN.appln_filing_year,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.granted,
        TLS201_APPLN.nb_inventors,
        TLS201_APPLN.nb_applicants,
    ).filter(*REAL, TLS201_APPLN.docdb_family_id.in_(part)))


def fetch_appln_subclasses(ids):
    "4-character CPC subclasses of each APPLICATION (no Y tags); IPC for the applications without CPC."
    cpc = in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id,
        TLS224_APPLN_CPC.cpc_class_symbol.label("symbol"),
    ).join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
     .filter(*REAL, TLS201_APPLN.docdb_family_id.in_(part)).distinct())
    cpc["code"] = cpc["symbol"].astype(str).str.replace(" ", "", regex=False).str[:4]
    cpc = cpc[~cpc["code"].str.startswith("Y") & (cpc["code"].str.len() == 4)][["docdb_family_id", "appln_id", "code"]].drop_duplicates()
    ipc = in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id,
        TLS209_APPLN_IPC.ipc_class_symbol.label("symbol"),
    ).join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
     .filter(*REAL, TLS201_APPLN.docdb_family_id.in_(part)).distinct())
    ipc["code"] = ipc["symbol"].astype(str).str.replace(" ", "", regex=False).str[:4]
    ipc = ipc[(ipc["code"].str.len() == 4) & ~ipc["appln_id"].isin(set(cpc["appln_id"]))][["docdb_family_id", "appln_id", "code"]]
    return pd.concat([cpc, ipc.drop_duplicates()], ignore_index=True)


def fetch_forward_cites(ids):
    return in_chunks(ids, lambda part: db.query(
        TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.label("docdb_family_id"),
        TLS228_DOCDB_FAM_CITN.docdb_family_id.label("citing_family"),
        func.min(TLS201_APPLN.earliest_filing_year).label("citing_year"),
    ).join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == TLS228_DOCDB_FAM_CITN.docdb_family_id)
     .filter(*REAL,
             TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.in_(part),
             TLS228_DOCDB_FAM_CITN.docdb_family_id != TLS228_DOCDB_FAM_CITN.cited_docdb_family_id)
     .group_by(TLS228_DOCDB_FAM_CITN.cited_docdb_family_id, TLS228_DOCDB_FAM_CITN.docdb_family_id))


apps = cached("applications", fetch_applications, FAMILIES)
appln_codes = cached("appln_subclasses", fetch_appln_subclasses, FAMILIES)
fwd = cached("forward_citations", fetch_forward_cites, FAMILIES)

# Dates of each application relative to the first priority of its family (used by every point-in-time feature)
apps_dt = apps.copy()
apps_dt["appln_auth"] = apps_dt["appln_auth"].astype(str).str.strip()
apps_dt["earliest_dt"] = pd.to_datetime(apps_dt["earliest_filing_date"], errors="coerce")
apps_dt["filing_dt"] = pd.to_datetime(apps_dt["appln_filing_date"], errors="coerce")
apps_dt["earliest_dt"] = apps_dt["earliest_dt"].where(apps_dt["earliest_dt"].dt.year < 9000)
apps_dt["filing_dt"] = apps_dt["filing_dt"].where(apps_dt["filing_dt"].dt.year < 9000)
fam_first_dt = apps_dt.groupby("docdb_family_id")["earliest_dt"].min()
apps_dt["days_from_prio"] = (apps_dt["filing_dt"] - apps_dt["docdb_family_id"].map(fam_first_dt)).dt.days
early_apps = early_rows(apps_dt, max_days=366)          # the applications filed within the priority year (Paris Convention)

# Codes known at filing: those of the applications filed within 12 months. A family whose early applications carry no
# code (rare) takes the codes of its first application with codes, so that it is not left without a subclass.
early_codes = appln_codes[appln_codes["appln_id"].isin(set(early_apps["appln_id"]))]
missing_fams = set(net.index) - set(early_codes["docdb_family_id"])
if missing_fams:
    _first_coded = (appln_codes[appln_codes["docdb_family_id"].isin(missing_fams)]
                    .merge(apps_dt[["appln_id", "filing_dt"]], on="appln_id").sort_values("filing_dt")
                    .drop_duplicates("docdb_family_id")[["docdb_family_id", "appln_id"]])
    early_codes = pd.concat([early_codes, appln_codes[appln_codes["appln_id"].isin(set(_first_coded["appln_id"]))]])
fam_codes = early_codes[["docdb_family_id", "code"]].drop_duplicates()

main_sub = (fam_codes.groupby(["docdb_family_id", "code"]).size().rename("n").reset_index()
            .sort_values(["docdb_family_id", "n", "code"], ascending=[True, False, True])
            .drop_duplicates("docdb_family_id").set_index("docdb_family_id")["code"])
net["main_subclass"] = net.index.map(main_sub).fillna("none")

# Office of first filing (national, not WO), the office group of the target cohort
first_off = (apps_dt[apps_dt["appln_auth"] != "WO"].sort_values(["docdb_family_id", "filing_dt", "appln_auth"])
             .drop_duplicates("docdb_family_id").set_index("docdb_family_id")["appln_auth"])
net["first_office"] = net.index.map(first_off).fillna("OTHER")
net["first_office_group"] = net["first_office"].where(net["first_office"].isin(["CN", "US", "JP", "KR", "EP"]), "OTHER")

# 5-year forward citations target
fwd_m = fwd.merge(net[["priority_year"]], left_on="docdb_family_id", right_index=True)
in_5y = fwd_m["citing_year"] <= fwd_m["priority_year"] + HORIZON
net["fwd_cites_5y"] = net.index.map(fwd_m[in_5y].groupby("docdb_family_id")["citing_family"].nunique()).fillna(0).astype(int)

# The target: top 10% of the cohort priority year x main subclass x office of first filing (coarser when the cohort has
# fewer than MIN_COHORT families). The extension families are not labelled: they never enter training or testing.
labelled_mask = net["priority_year"].between(YEAR_MIN, LAST_LABEL_YEAR) & ~net["forced_extension"]
net["valuable"] = np.nan
net.loc[labelled_mask, "valuable"], net.loc[labelled_mask, "cohort"] = label_top_decile_hierarchical(
    net.loc[labelled_mask], value_col="fwd_cites_5y",
    levels=[("priority_year", "main_subclass", "first_office_group"), ("priority_year", "main_subclass"), ("priority_year",)],
    top_share=TOP_SHARE, min_cohort=MIN_COHORT,
)
_lvl = net.loc[labelled_mask, "cohort"].astype(str).str.count(r"\|")
print("cohort level used:", _lvl.map({2: "year x subclass x office", 1: "year x subclass", 0: "year"}).value_counts().to_dict())

target_summary = net.loc[labelled_mask].groupby("priority_year").agg(
    families=("valuable", "size"),
    humanoid=("in_sector_v3", "sum"),
    top10_rate_pct=("valuable", lambda s: s.mean() * 100),
    median_cites_5y=("fwd_cites_5y", "median"),
    p90_cites_5y=("fwd_cites_5y", lambda s: s.quantile(0.90)),
)
by_office_target = net.loc[labelled_mask].groupby("first_office_group")["valuable"].agg(families="size", top10_rate="mean")
print(f"Labelled families ({YEAR_MIN}-{LAST_LABEL_YEAR}): {int(labelled_mask.sum()):,} "
      f"(valuable top-10%: {int(net['valuable'].sum()):,} = {net['valuable'].mean():.1%})")
display(target_summary.round(2))
print("top-10% rate by office of first filing (about 10% in every office: the office habits are out of the target):")
display(by_office_target.round(3))

> **⚠ Rerun needed (review of 28 Nov 2026).** The code above this reading changed; the numbers below are those of the previous PROD run and must be checked, and rewritten if they moved, after *Restart and Run All* on PROD.

**Our reading of the target.** *To rewrite*: the number of labelled families and positives, the share of families labelled at each cohort level, and the top-10% rate by office (it should be close to 10% everywhere, which is the point of the office cohort).

## 3. Point-in-time features (what is known 18 months after the first priority) + $t_0 + 2\text{y}$ diagnostics

A feature can be used only if it would have been known when we score a family. The first version took some features from the whole later life of the family: the backward citations of TLS228 include those added years later by the examiners of every office where the family was extended, and the CPC codes of every later family member. These grow with the later size of the family, which is itself linked to its citations: a leak. Now:

- **offices, PCT, inventors, applicants and codes** come from the applications filed within 12 months of the first priority (the priority year of the Paris Convention);
- **backward citations, originality, technology cycle time, examiner X/Y/I** come from the publications published within 18 months of the first priority (TLS211 dates, TLS212 citations, TLS215 categories): roughly the first publication with its search report;
- families of 2024–2025 have not all reached 18 months by the date of the data: Section 3 counts them, Section 6 flags them.

Each feature is registered with its source, its set (`classic` benchmark vs. `ours`) and its expected direction for the zero-training scorecard (`Theory 6`, p. 25).

In [ ]:
SIGNALS = {}
signals = pd.DataFrame(index=net.index)


def register(name, series, group, timing, direction, source):
    signals[name] = pd.Series(series).reindex(net.index).astype(float)
    SIGNALS[name] = {"set": group, "timing": timing, "direction": direction, "source": source}


# --- Pillar 1: classic ex-ante metadata, from the applications filed within 12 months (apps_dt, early_apps: Section 2) ---
early_nat = early_apps[early_apps["appln_auth"] != "WO"]
register("early_offices_12m", early_nat.groupby("docdb_family_id")["appln_auth"].nunique().reindex(net.index).fillna(1),
         "classic", "static", +1, "Harhoff et al. (2003)")
register("has_pct", early_apps[early_apps["appln_auth"] == "WO"].groupby("docdb_family_id").size().gt(0).reindex(net.index).fillna(False).astype(float),
         "classic", "static", +1, "Theory 6 p. 12 (PCT route)")
register("n_inventors", early_apps.groupby("docdb_family_id")["nb_inventors"].max().reindex(net.index).fillna(0),
         "classic", "static", +1, "Theory 5 p. 23; Lee et al. (2018)")
register("n_applicants", early_apps.groupby("docdb_family_id")["nb_applicants"].max().reindex(net.index).fillna(1),
         "classic", "static", +1, "Theory 5 p. 43 (co-applicants)")
register("scope_cpc4", fam_codes.groupby("docdb_family_id")["code"].nunique().reindex(net.index).fillna(1),
         "classic", "static", +1, "Lerner (1994); Lai et al. (2026)")

# University co-applicant flag (TLS207 + TLS206), on the applications filed within 12 months
def fetch_univ_flag(ids):
    df = in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS206_PERSON.psn_sector,
    ).join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
     .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
     .filter(*REAL, TLS207_PERS_APPLN.applt_seq_nr > 0, TLS201_APPLN.docdb_family_id.in_(part)).distinct())
    df["is_univ"] = df["psn_sector"].astype(str).str.upper().str.contains("UNIVERSITY")
    return df[["docdb_family_id", "appln_id", "is_univ"]]


univ_rows = cached("univ_coapp_by_appln", fetch_univ_flag, FAMILIES)
univ_flag = univ_rows[univ_rows["appln_id"].isin(set(early_apps["appln_id"]))].groupby("docdb_family_id")["is_univ"].any().astype(float)
register("has_university", univ_flag.reindex(net.index).fillna(0.0), "classic", "static", 0, "Step 1 & 2 (academic vs corporate)")

# First filing office controls (Lemley & Sampat 2012): net["first_office"] from Section 2
for _off in ("CN", "US", "JP", "KR", "EP"):
    register(f"office_{_off}", (net["first_office"] == _off).astype(float), "classic", "static", 0, "Lemley & Sampat (2012)")

print(f"Pillar 1 (classic ex-ante, applications filed within 12 months) registered: {len(SIGNALS)} features")

In [ ]:
# --- Pillar 2: backward citations, originality, technology cycle time and recombination, POINT IN TIME ---
# The publications of the universe with their dates; only those published within FEATURE_AGE_DAYS of the first priority
# are used: their citations are what an analyst would have seen at that age (the first search report).
def fetch_publications(_):
    aids = apps["appln_id"].astype(int).unique().tolist()
    return in_chunks(aids, lambda part: db.query(
        TLS211_PAT_PUBLN.pat_publn_id, TLS211_PAT_PUBLN.appln_id, TLS211_PAT_PUBLN.publn_auth, TLS211_PAT_PUBLN.publn_date
    ).filter(TLS211_PAT_PUBLN.appln_id.in_(part)))


pubs_all = cached("publications", fetch_publications, None)
pubs_all["docdb_family_id"] = pubs_all["appln_id"].map(apps.drop_duplicates("appln_id").set_index("appln_id")["docdb_family_id"])
pubs_all["publn_dt"] = pd.to_datetime(pubs_all["publn_date"], errors="coerce")
pubs_all["publn_dt"] = pubs_all["publn_dt"].where(pubs_all["publn_dt"].dt.year < 9000)
pubs_all["days_from_prio"] = (pubs_all["publn_dt"] - pubs_all["docdb_family_id"].map(fam_first_dt)).dt.days
DATA_CUT = pubs_all["publn_dt"].max()                    # the last publication date of the data (edition of PATSTAT)
early_pubs = pubs_all[pubs_all["days_from_prio"].between(-1, FEATURE_AGE_DAYS)]
net["feature_window_complete"] = (net.index.map(fam_first_dt) + pd.Timedelta(days=FEATURE_AGE_DAYS)) <= DATA_CUT
print(f"publications: {len(pubs_all):,}; within {FEATURE_AGE_DAYS} days of priority: {len(early_pubs):,}; data cut {DATA_CUT.date()}")
print("families whose 18-month window is complete, by priority year (recent years):",
      net.groupby("priority_year")["feature_window_complete"].mean().loc[2021:].round(2).to_dict())


def fetch_backward_cites_pit(_):
    pid = early_pubs["pat_publn_id"].astype(int).unique().tolist()
    c = in_chunks(pid, lambda part: db.query(TLS212_CITATION.pat_publn_id, TLS212_CITATION.cited_pat_publn_id)
                  .filter(TLS212_CITATION.pat_publn_id.in_(part), TLS212_CITATION.cited_pat_publn_id > 0).distinct())
    if not len(c):
        return pd.DataFrame(columns=["docdb_family_id", "cited_family", "cited_year", "cited_subclass"])
    c["docdb_family_id"] = c["pat_publn_id"].map(early_pubs.drop_duplicates("pat_publn_id").set_index("pat_publn_id")["docdb_family_id"])
    cited = in_chunks(c["cited_pat_publn_id"].astype(int).unique().tolist(), lambda part: db.query(
        TLS211_PAT_PUBLN.pat_publn_id.label("cited_pat_publn_id"),
        TLS201_APPLN.docdb_family_id.label("cited_family"),
        TLS201_APPLN.earliest_filing_year.label("cited_year"),
    ).join(TLS201_APPLN, TLS201_APPLN.appln_id == TLS211_PAT_PUBLN.appln_id).filter(TLS211_PAT_PUBLN.pat_publn_id.in_(part)))
    b = (c.merge(cited, on="cited_pat_publn_id")
          .groupby(["docdb_family_id", "cited_family"], as_index=False)["cited_year"].min())
    b = b[b["docdb_family_id"] != b["cited_family"]]
    c_ipc = in_chunks(b["cited_family"].astype(int).unique().tolist(), lambda part: db.query(
        TLS201_APPLN.docdb_family_id.label("cited_family"),
        TLS209_APPLN_IPC.ipc_class_symbol.label("symbol"),
    ).join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
     .filter(*REAL, TLS201_APPLN.docdb_family_id.in_(part)).distinct())
    c_ipc["cited_subclass"] = c_ipc["symbol"].astype(str).str.replace(" ", "", regex=False).str[:4]
    c_ipc = (c_ipc[c_ipc["cited_subclass"].str.len() == 4]
             .groupby(["cited_family", "cited_subclass"]).size().rename("n").reset_index()
             .sort_values(["cited_family", "n", "cited_subclass"], ascending=[True, False, True])
             .drop_duplicates("cited_family")[["cited_family", "cited_subclass"]])
    return b.merge(c_ipc, on="cited_family", how="left")


bwd = cached("backward_citations_pit", fetch_backward_cites_pit, None)
orig_tct = compute_originality_and_tct(bwd, net["priority_year"])
register("n_backward_cites", orig_tct["n_bwd_cited_fams"].reindex(net.index).fillna(0.0),
         "classic", "static", +1, "Harhoff et al. (2003); first search report only")
register("originality", orig_tct["originality"],
         "ours", "static", +1, "Trajtenberg et al. (1997); Lai et al. (2026)")
register("tech_cycle_time", orig_tct["tech_cycle_time"],
         "ours", "static", -1, "Lai et al. (2026) (shorter = faster frontier)")

# Causal recombination novelty (NR, Fleming 2001; Verhoeven et al. 2016), on the codes known at filing
nr_df = compute_causal_recombination_novelty(fam_codes, net["priority_year"])
register("novel_pair_share_nr", nr_df["novel_pair_share_nr"].reindex(net.index).fillna(0.0),
         "ours", "static", +1, "Fleming (2001); Verhoeven et al. (2016)")
register("has_novel_pair_nr", nr_df["has_novel_pair_nr"].reindex(net.index).fillna(0.0),
         "ours", "static", +1, "Fleming (2001); Theory 6 p. 18")

# Causal hot-subfield momentum (strictly before filing year t)
mom = compute_causal_subfield_momentum(net, window=3)
register("subfield_momentum", mom, "ours", "static", +1, "Theory 6 p. 23 (hot subfields)")

print(f"Pillar 2 + subfield momentum registered | families with a backward citation at 18 months: "
      f"{(signals['n_backward_cites'] > 0).mean():.1%} | novel pair share > 0: {(signals['has_novel_pair_nr'] > 0).mean():.1%}")

In [ ]:
# --- Pillar 3: examiner search reports (TLS215 X/Y/I, all offices), on the publications of the first 18 months ---
def fetch_search_reports_pit(_):
    pid = early_pubs["pat_publn_id"].astype(int).unique().tolist()
    cats = in_chunks(pid, lambda part: db.query(
        TLS215_CITN_CATEG.pat_publn_id, TLS215_CITN_CATEG.citn_categ
    ).filter(TLS215_CITN_CATEG.pat_publn_id.in_(part)))
    if not len(cats):
        return pd.DataFrame(columns=["docdb_family_id", "has_report", "has_xy", "xy_share", "n_cats"])
    cats = cats.merge(early_pubs[["pat_publn_id", "docdb_family_id"]].drop_duplicates(), on="pat_publn_id")
    cats["is_xy"] = cats["citn_categ"].astype(str).str.contains("[XYI]", regex=True)
    return (cats.groupby("docdb_family_id")
            .agg(has_report=("is_xy", lambda s: 1.0),
                 has_xy=("is_xy", "max"),
                 xy_share=("is_xy", "mean"),
                 n_cats=("is_xy", "size"))
            .reset_index())


sr = cached("search_reports_tls215_pit", fetch_search_reports_pit, None).set_index("docdb_family_id")
register("has_search_report", sr["has_report"].reindex(net.index).fillna(0.0),
         "ours", "static", 0, "TLS215 coverage flag")
register("examiner_has_xy", sr["has_xy"].astype(float),
         "ours", "static", -1, "Czarnitzki et al. (2011); Theory 6 p. 12 (missing if no report)")
register("examiner_xy_share", sr["xy_share"],
         "ours", "static", -1, "Czarnitzki et al. (2011); Lemley & Sampat (2012)")

# Check on 2015-2017 labelled families: do families WITHOUT X/Y get cited more?
_sr_check = net[net["priority_year"].between(2015, 2017) & net["valuable"].notna()].join(signals[["has_search_report", "examiner_has_xy"]])
_sr_with = _sr_check[_sr_check["has_search_report"] == 1]
if len(_sr_with):
    print("Examiner X/Y check on 2015-2017 families with a search report within 18 months:")
    display(_sr_with.groupby("examiner_has_xy").agg(
        families=("fwd_cites_5y", "size"),
        mean_cites_5y=("fwd_cites_5y", "mean"),
        top10_rate_pct=("valuable", lambda s: s.mean() * 100),
    ).round(2))

> **⚠ Rerun needed (review of 28 Nov 2026).** The code above this reading changed; the numbers below are those of the previous PROD run and must be checked, and rewritten if they moved, after *Restart and Run All* on PROD. The search reports are now those of the first 18 months only, and the international target is gone.

**Our reading of the examiner's opinion (`TLS215`).** On `2015–2017` families with search reports, families with an `X`/`Y`/`I` citation (`n = 3,134`) initially show slightly higher *unadjusted* mean 5-year citations (`8.00` vs. `6.43`, top-10% share `13.2%` vs. `10.3%`, international rate `30.7%` vs. `15.9%`) because crowded, commercially active filings attract both heavier filing abroad and tighter examiner scrutiny. Once we control for office, backward citations, and scope in the multivariate model (Section 5), `examiner_xy_share` carries the expected **negative sign (`-0.054`)**: heavier blocking prior art against an application signals narrower novelty.


In [ ]:
# --- Pillar 4: Sector-only Text Novelty (Arts zero-leakage + Kelly SBERT), Readability & OpenAlex PaECTER Science ---
SECTOR_ONLY = []

# 4a. Kelly et al. (2021) semantic novelty from Notebook 21
if TEXT_NOVELTY_FILE.exists():
    _t21 = pd.read_csv(TEXT_NOVELTY_FILE).drop_duplicates("docdb_family_id").set_index("docdb_family_id")
    _col = "novelty_rank_in_year" if "novelty_rank_in_year" in _t21.columns else _t21.columns[0]
    register("kelly_semantic_novelty_rank", _t21[_col], "ours", "static", +1, "Kelly et al. (2021); Notebook 21")
    SECTOR_ONLY.append("kelly_semantic_novelty_rank")

# 4b. Arts, Hou & Gomez (2021) ZERO-LEAKAGE lexical novelty + Lai et al. (2026) readability on sector abstracts
_abs_df = pd.DataFrame()
for _cand in (Path("outputs/t2_text/sector_abstracts_PROD_rule_v3.csv"), Path("outputs/step0/prod/02_net_abstracts.csv.gz")):
    if _cand.exists():
        _tmp = pd.read_csv(_cand)
        _acol = next((c for c in ("abstract", "appln_abstract", "abstract_text") if c in _tmp.columns), None)
        if _acol and "docdb_family_id" in _tmp.columns:
            _abs_df = (_tmp[_tmp["docdb_family_id"].isin(sector_ids)][["docdb_family_id", _acol]]
                       .dropna().drop_duplicates("docdb_family_id").rename(columns={_acol: "text"}))
            break

if len(_abs_df):
    _abs_df = _abs_df.merge(sector_families[["docdb_family_id", "priority_year"]], on="docdb_family_id")
    arts_df = compute_arts_lexical_novelty(_abs_df, text_col="text", year_col="priority_year", lookback_years=5)
    arts_df.to_csv(OUT_DIR / "arts_lexical_novelty.csv")
    register("arts_novelty_rank", arts_df["arts_novelty_rank_in_year"], "ours", "static", +1, "Arts, Hou & Gomez (2021) — zero leakage")
    register("abstract_long_word_share", arts_df["abstract_long_word_share"], "ours", "static", +1, "Lai et al. (2026) readability")
    SECTOR_ONLY += ["arts_novelty_rank", "abstract_long_word_share"]

# 4c. OpenAlex + PaECTER causal implicit science linkage from Step 3 Section 3.6
if OPENALEX_FILE.exists():
    _oa = pd.read_csv(OPENALEX_FILE).drop_duplicates("docdb_family_id").set_index("docdb_family_id")
    register("openalex_science_rank", _oa["science_rank_in_year"], "ours", "static", +1, "Theory 6 pp. 21-22 (OpenAlex + PaECTER)")
    register("openalex_fundamental_match", _oa["fundamental_match"].astype(float), "ours", "static", +1, "Theory 6 p. 22 (fast-rising papers)")
    SECTOR_ONLY += ["openalex_science_rank", "openalex_fundamental_match"]

# --- Pillar 5: t0 + 2 years network signals (Mariani et al. 2019; Funk & Owen-Smith 2017): a DIAGNOSTIC only ---
# Their early citations are part of the 5-year citations of the target, and the families we score (2022-2025) do not
# have two years of citations yet: they are never used to score firms and never quoted as the model's performance.
dyn_2y = cached("dynamic_network_2y_pit", lambda _: compute_dynamic_network_at_age(fwd, bwd, net["priority_year"], age_limit=2), None)
if "docdb_family_id" in dyn_2y.columns:
    dyn_2y = dyn_2y.set_index("docdb_family_id")
else:
    dyn_2y.index = net.index[:len(dyn_2y)]
DYNAMIC_2Y = ["early_cites_2y", "pagerank_age_ratio_2y", "cd_index_2y"]
register("early_cites_2y", dyn_2y["early_cites_2y"], "dynamic_2y", "dynamic", +1, "Lai et al. (2026) t0+2y")
register("pagerank_age_ratio_2y", dyn_2y["pagerank_age_ratio_2y"], "dynamic_2y", "dynamic", +1, "Mariani, Medo & Lafond (2019)")
register("cd_index_2y", dyn_2y["cd_index_2y"], "dynamic_2y", "dynamic", +1, "Funk & Owen-Smith (2017)")

sig_summary = pd.DataFrame(SIGNALS).T
sig_summary["coverage (%)"] = (signals.notna().mean() * 100).round(1)
print(f"Total registered signals: {len(SIGNALS)} (sector-only text/science: {len(SECTOR_ONLY)} | dynamic t0+2y: {len(DYNAMIC_2Y)})")
display(sig_summary)


## 4. Out-of-time backtest: the validation years choose, the test years report

Following **Theory 6 (pp. 35–37)**, for each year $T$ we train strictly on $[T-5, T-1]$ and test on $T$, on the real unbalanced distribution. The first version chose the model on the same years it reported; now:

- **validation years 2014–2015**: we choose the model (logistic, random forest, boosting) for our feature set;
- **test years 2016–2017**: we report PR-AUC, ROC-AUC, Precision/Recall/Lift@10% of the chosen model, never used to choose anything;
- **the gain over the classic benchmark** is measured with a **paired bootstrap** on the same test families (difference of PR-AUC, 95% CI, one-sided p): two separate intervals that overlap do not tell whether one model beats the other;
- baselines on the same test families: random, the zero-training scorecard, single literature signals;
- the $t_0 + 2\text{y}$ model is shown as a **diagnostic** (how much the first two years of citations add), not as our performance: it uses part of the target and cannot score recent families.

In [ ]:
CLASSIC = [s for s, d in SIGNALS.items() if d["set"] == "classic"]
OURS_UNIV = [s for s, d in SIGNALS.items() if d["set"] == "ours" and s not in SECTOR_ONLY]
BENCH, PROPOSED, DYNAMIC = "1_benchmark_t0 (classic)", "2_proposed_t0 (classic + ours)", "3_t0+2y diagnostic (+ PageRank/CD)"
FEATURE_SETS = {BENCH: CLASSIC, PROPOSED: CLASSIC + OURS_UNIV, DYNAMIC: CLASSIC + OURS_UNIV + DYNAMIC_2Y}
y_model = net["valuable"]                                  # NaN for the extension families: never trained or tested on

preds = run_rolling_oot_backtest(
    signals, y_model, net["priority_year"], net["in_sector_v3"],
    feature_sets=FEATURE_SETS, test_years=BACKTEST_YEARS, lookback=LOOKBACK, random_state=RANDOM_STATE,
)
preds["split"] = np.where(preds["test_year"].isin(TEST_YEARS), "test", "validation")
preds.to_csv(OUT_DIR / "backtest_predictions.csv", index=False)

# Zero-training Rule-Based Scorecard
_rule_dirs = {k: d["direction"] for k, d in SIGNALS.items() if d["direction"] != 0 and k not in SECTOR_ONLY and k not in DYNAMIC_2Y}
signals["rule_based_t0"] = rule_based_future_score(signals, _rule_dirs, year_series=net["priority_year"])


def eval_table(pred_df, subset_label="All robotics", only_sector=False):
    rows = []
    sub_all = pred_df[pred_df["in_sector"]] if only_sector else pred_df
    for (fset, mod), g in sub_all.groupby(["feature_set", "model"]):
        m = precision_recall_lift_at_k(g["y"], g["score"], share=TOP_SHARE)
        pr_pt, pr_lo, pr_hi = bootstrap_metric_ci(g["y"], g["score"], "PR_AUC", n_boot=300, seed=RANDOM_STATE)
        rows.append({
            "scope": subset_label, "feature_set": fset, "model": mod,
            "n_test": len(g), "positives": int(g["y"].sum()), "base_rate (%)": m["base_rate"] * 100,
            "PR_AUC": pr_pt, "PR_AUC_95CI": f"[{pr_lo:.3f}, {pr_hi:.3f}]",
            "ROC_AUC": m["ROC_AUC"], "P@10% (%)": m["P_at_10pct"] * 100,
            "R@10% (%)": m["R_at_10pct"] * 100, "Lift@10%": m["Lift_at_10pct"],
        })
    return pd.DataFrame(rows).sort_values(["feature_set", "PR_AUC"], ascending=[True, False])


val_preds, test_preds = preds[preds["split"] == "validation"], preds[preds["split"] == "test"]
bt_val = eval_table(val_preds, "Robotics universe · validation", only_sector=False)
bt_all = eval_table(test_preds, "Robotics universe · test", only_sector=False)       # the numbers of the deck
bt_sec = eval_table(test_preds, "Humanoid sector only · test", only_sector=True)

# The model is chosen on the VALIDATION years only (our feature set)
BEST_MODEL_NAME = str(bt_val[bt_val["feature_set"] == PROPOSED].sort_values("PR_AUC", ascending=False).iloc[0]["model"])
print(f"model chosen on the validation years {VALIDATION_YEARS}: {BEST_MODEL_NAME}")

# Baselines on the same TEST families
test_fids = test_preds["docdb_family_id"].drop_duplicates().to_numpy()
y_test_all = net.loc[test_fids, "valuable"].astype(int).to_numpy()
base_rows = []
for bname, bcol in [
    ("Rule-Based Scorecard (t0, zero training)", "rule_based_t0"),
    ("Single signal: early_offices_12m", "early_offices_12m"),
    ("Single signal: originality", "originality"),
    ("Single signal: novel_pair_share_nr", "novel_pair_share_nr"),
    ("Diagnostic: PageRank age-rescaled (t0+2y)", "pagerank_age_ratio_2y"),
]:
    if bcol not in signals.columns:
        continue
    s_vec = signals.loc[test_fids, bcol].to_numpy()
    m = precision_recall_lift_at_k(y_test_all, s_vec, share=TOP_SHARE)
    base_rows.append({
        "scope": "Robotics universe · test", "feature_set": "0_baselines", "model": bname,
        "n_test": int(np.isfinite(s_vec).sum()), "positives": int(y_test_all.sum()), "base_rate (%)": m["base_rate"] * 100,
        "PR_AUC": m["PR_AUC"], "PR_AUC_95CI": "—", "ROC_AUC": m["ROC_AUC"],
        "P@10% (%)": m["P_at_10pct"] * 100, "R@10% (%)": m["R_at_10pct"] * 100, "Lift@10%": m["Lift_at_10pct"],
    })
baselines_df = pd.DataFrame(base_rows).sort_values("PR_AUC", ascending=False)
backtest_summary = pd.concat([baselines_df, bt_all, bt_sec, bt_val], ignore_index=True)
backtest_summary.to_csv(OUT_DIR / "backtest_summary.csv", index=False)
print(f"=== Out-of-time backtest: TEST years {TEST_YEARS} (the validation rows are at the bottom) ===")
display(backtest_summary.round(3))


# The honest comparison: paired bootstrap of the PR-AUC difference on the same TEST families, chosen model
def paired(pred_df, set_a, set_b, model, label, only_sector=False):
    d = pred_df[pred_df["in_sector"]] if only_sector else pred_df
    a = d[(d["feature_set"] == set_a) & (d["model"] == model)].set_index("docdb_family_id")
    b = d[(d["feature_set"] == set_b) & (d["model"] == model)].set_index("docdb_family_id")
    j = a[["y", "score"]].join(b[["score"]], rsuffix="_b", how="inner")
    r = paired_bootstrap_diff(j["y"], j["score"], j["score_b"], "PR_AUC", n_boot=1000, seed=RANDOM_STATE)
    return {"comparison": label, **r}


paired_tab = pd.DataFrame([
    paired(test_preds, BENCH, PROPOSED, BEST_MODEL_NAME, f"ours vs classic ({BEST_MODEL_NAME}), robotics"),
    paired(test_preds, BENCH, PROPOSED, BEST_MODEL_NAME, f"ours vs classic ({BEST_MODEL_NAME}), humanoid only", only_sector=True),
])
_ours = test_preds[(test_preds["feature_set"] == PROPOSED) & (test_preds["model"] == BEST_MODEL_NAME)].set_index("docdb_family_id")
_rb = paired_bootstrap_diff(_ours["y"], signals.loc[_ours.index, "rule_based_t0"], _ours["score"], "PR_AUC", n_boot=1000, seed=RANDOM_STATE)
paired_tab = pd.concat([paired_tab, pd.DataFrame([{"comparison": "ours vs zero-training scorecard, robotics", **_rb}])], ignore_index=True)
paired_tab.to_csv(OUT_DIR / "paired_comparison_test.csv", index=False)
print("PR-AUC difference on the TEST years (paired bootstrap, 1,000 draws): "
      "a gain is shown only if its 95% interval is above 0")
display(paired_tab.round(3))


def _plot_backtest():
    best_by_set = bt_all[bt_all["model"] == BEST_MODEL_NAME]
    comp = pd.concat([baselines_df[["model", "PR_AUC", "Lift@10%"]].rename(columns={"model": "label"}),
                      best_by_set.assign(label=best_by_set["feature_set"] + " · " + best_by_set["model"])[["label", "PR_AUC", "Lift@10%"]]],
                     ignore_index=True).sort_values("PR_AUC", ascending=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.2, 4.6))
    cols = ["#e67e22" if "proposed" in l else ("#2c7bb6" if "benchmark" in l else "#95a5a6") for l in comp["label"]]
    ax1.barh(comp["label"], comp["PR_AUC"], color=cols)
    ax1.axvline(y_test_all.mean(), color="#c0392b", ls="--", lw=1.2, label=f"Random ({y_test_all.mean():.3f})")
    for k, v in enumerate(comp["PR_AUC"]):
        ax1.text(v + 0.006, k, f"{v:.3f}", va="center", fontsize=8.5)
    ax1.set_xlabel(f"PR-AUC (out of time, test years {TEST_YEARS[0]}-{TEST_YEARS[-1]})")
    ax1.set_title("PR-AUC vs. literature baselines (grey = baselines and diagnostics)")
    ax1.legend(loc="lower right", fontsize=8)
    ax1.spines[["top", "right"]].set_visible(False)
    ax2.barh(comp["label"], comp["Lift@10%"], color=cols)
    ax2.axvline(1.0, color="#c0392b", ls="--", lw=1.2, label="Random (1.0×)")
    for k, v in enumerate(comp["Lift@10%"]):
        ax2.text(v + 0.06, k, f"{v:.2f}×", va="center", fontsize=8.5)
    ax2.set_yticks([])
    ax2.set_xlabel("Lift in the top-10% shortlist (× over base rate)")
    ax2.set_title("Precision lift in the top 10%")
    ax2.legend(loc="lower right", fontsize=8)
    ax2.spines[["top", "right"]].set_visible(False)
    plt.suptitle(f"Section 4 · Out-of-time backtest, test years (model chosen on {VALIDATION_YEARS})", y=1.02, fontweight="bold")
    plt.tight_layout()
    plt.show()


_plot_backtest()
del _plot_backtest

> **⚠ Rerun needed (review of 28 Nov 2026).** The code above this reading changed; the numbers below are those of the previous PROD run and must be checked, and rewritten if they moved, after *Restart and Run All* on PROD.

**Our reading of the backtest.** *To rewrite from the test-year rows and the paired table*: the model chosen on the validation years; its PR-AUC, ROC-AUC and Lift@10% on the test years against the base rate and the baselines; and **whether the gain over the classic benchmark is real**: say it only if the paired 95% interval of the difference is above zero, otherwise write that our signals add no measurable gain over the classic metadata on the test years. The $t_0+2y$ row is a diagnostic (early citations are part of the target), never our performance.

In [ ]:
# Humanoid-sector-only ablation: does adding Arts zero-leakage novelty, Kelly SBERT novelty & OpenAlex science lift PR-AUC?
# (The "international" secondary target of the first version is dropped: early_offices_12m implied it.)
if SECTOR_ONLY:
    _sec_idx = net.index[net["in_sector_v3"]]
    _sec_sets = {
        "sector_meta_t0": CLASSIC + OURS_UNIV,
        "sector_meta + Arts_zero_leakage": CLASSIC + OURS_UNIV + [c for c in ("arts_novelty_rank", "abstract_long_word_share") if c in SECTOR_ONLY],
        "sector_meta + Kelly_SBERT": CLASSIC + OURS_UNIV + [c for c in ("kelly_semantic_novelty_rank",) if c in SECTOR_ONLY],
        "sector_meta + OpenAlex_PaECTER": CLASSIC + OURS_UNIV + [c for c in ("openalex_science_rank", "openalex_fundamental_match") if c in SECTOR_ONLY],
        "sector_ALL_multimodal_t0": CLASSIC + OURS_UNIV + SECTOR_ONLY,
    }
    preds_sec_abl = run_rolling_oot_backtest(
        signals.loc[_sec_idx], y_model.loc[_sec_idx], net.loc[_sec_idx, "priority_year"],
        net.loc[_sec_idx, "in_sector_v3"], feature_sets=_sec_sets, test_years=BACKTEST_YEARS, lookback=LOOKBACK, random_state=RANDOM_STATE,
    )
    preds_sec_abl = preds_sec_abl[preds_sec_abl["test_year"].isin(TEST_YEARS)]
    sec_ablation = eval_table(preds_sec_abl, "Humanoid sector ablation · test", only_sector=True)
    sec_ablation.to_csv(OUT_DIR / "sector_text_science_ablation.csv", index=False)
    print("\n=== Humanoid-sector-only ablation, test years: Arts lexical vs. Kelly SBERT vs. OpenAlex PaECTER ===")
    display(sec_ablation.round(3))
    _abl = pd.DataFrame([paired(preds_sec_abl.assign(in_sector=True), "sector_meta_t0", s, BEST_MODEL_NAME, f"{s} vs sector_meta_t0")
                         for s in _sec_sets if s != "sector_meta_t0"])
    print("paired PR-AUC differences against the metadata-only model (same humanoid test families):")
    display(_abl.round(3))

> **⚠ Rerun needed (review of 28 Nov 2026).** The code above this reading changed; the numbers below are those of the previous PROD run and must be checked, and rewritten if they moved, after *Restart and Run All* on PROD.

**Our reading of the humanoid text and science ablation.** *To rewrite*: on the humanoid test families only (few positives, wide intervals), which text or science signal adds PR-AUC to the metadata model, **with the paired interval**. With about 30 positives, a difference whose interval includes zero is not a finding. (The international target of the first version is dropped, see Section 2.)

## 5. Interpretability: which signals actually predict breakthroughs? (Theory 6, p. 43)

To ensure our valuation model is not a black box, we report both:
1. **Standardized Logistic Regression coefficients** (exact direction and magnitude on the `t0` proposed feature set);
2. **Holdout Permutation Importance** (`scoring="average_precision"`, PR-AUC drop on the out-of-time 2017 holdout set) for our Gradient Boosting model.


In [ ]:
# Fit on 2012-2016, inspect interpretability on 2017 holdout (zero leakage)
_tr_imp = net["priority_year"].between(2012, 2016) & net["valuable"].notna()
_te_imp = (net["priority_year"] == 2017) & net["valuable"].notna()
_feats_t0 = CLASSIC + OURS_UNIV

_clfs = make_classifiers(random_state=RANDOM_STATE)
_log_m = _clfs["logistic"].fit(signals.loc[_tr_imp, _feats_t0], net.loc[_tr_imp, "valuable"].astype(int))
_hgb_m = _clfs["boosting"].fit(signals.loc[_tr_imp, _feats_t0], net.loc[_tr_imp, "valuable"].astype(int))

# Logistic standardized coefficients (first len(_feats_t0) correspond to the features before missingness indicators)
_coefs = _log_m.named_steps["model"].coef_[0][:len(_feats_t0)]
_perm = permutation_importance(
    _hgb_m, signals.loc[_te_imp, _feats_t0], net.loc[_te_imp, "valuable"].astype(int),
    scoring="average_precision", n_repeats=15, random_state=RANDOM_STATE,
)
importance_df = pd.DataFrame({
    "feature": _feats_t0,
    "group": [SIGNALS[f]["set"] for f in _feats_t0],
    "logistic_std_coef": _coefs,
    "boosting_perm_PR_AUC_drop": _perm.importances_mean,
    "perm_std": _perm.importances_std,
}).sort_values("boosting_perm_PR_AUC_drop", ascending=False)
importance_df.to_csv(OUT_DIR / "feature_importance.csv", index=False)
display(importance_df.round(4))

def _plot_importance():
    top = importance_df.head(14).sort_values("boosting_perm_PR_AUC_drop", ascending=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.8, 4.8))
    cols1 = ["#e67e22" if g == "ours" else "#2c7bb6" for g in top["group"]]
    ax1.barh(top["feature"], top["boosting_perm_PR_AUC_drop"], xerr=top["perm_std"], color=cols1, capsize=2)
    ax1.axvline(0, color="black", lw=0.8)
    ax1.set_xlabel("Holdout PR-AUC drop when permuted (2017 test)")
    ax1.set_title("Boosting Permutation Importance (Orange = Ours)")
    ax1.spines[["top", "right"]].set_visible(False)

    top_c = importance_df.reindex(top.index)
    cols2 = ["#27ae60" if c > 0 else "#c0392b" for c in top_c["logistic_std_coef"]]
    ax2.barh(top_c["feature"], top_c["logistic_std_coef"], color=cols2)
    ax2.axvline(0, color="black", lw=0.8)
    ax2.set_yticks([])
    ax2.set_xlabel("Standardized Logistic Regression coefficient (sign)")
    ax2.set_title("Direction of Effect (Green = +, Red = −)")
    ax2.spines[["top", "right"]].set_visible(False)
    plt.suptitle("Section 5 · Feature Importance & Direction (t0 Model)", y=1.02, fontweight="bold")
    plt.tight_layout()
    plt.show()

_plot_importance()
del _plot_importance


> **⚠ Rerun needed (review of 28 Nov 2026).** The code above this reading changed; the numbers below are those of the previous PROD run and must be checked, and rewritten if they moved, after *Restart and Run All* on PROD. The features are now point in time, so the importance of `n_backward_cites` and `scope_cpc4` (the two features most exposed to the leak) must be re-read.

**Our reading of feature importance (`Theory 6`, p. 43).** Holdout permutation importance (2017 test set) and standardized Logistic Regression signs agree on a clear, interpretable economic story:
1. **`n_backward_cites`** (`+0.105` PR-AUC drop, coef `+0.726`) and **`scope_cpc4`** (`+0.049`, `+0.353`) — broad, well-grounded inventions that bridge multiple technical subclasses;
2. **`tech_cycle_time`** (`+0.032`, coef **`-0.321`**) — **shorter technology cycle time** (building on fresher prior art) strongly increases breakthrough odds, matching Lai, Wei & Hwang (2026);
3. **`early_offices_12m`** (`+0.024`, `+0.493`) and **`office_US`** (`+0.008`, `+0.433`) — early international filing commitment signals high private valuation by the applicant;
4. **`n_inventors`** (`+0.003`, `+0.097`) and **`examiner_xy_share`** (`+0.001`, **`-0.054`**) — larger R&D teams help, while denser blocking prior art against the filing reduces breakthrough probability.


## 6. Scoring the recent families and the common set of firms

1. **The final model** (the one chosen on the validation years) is trained on $[2012, 2016]$ and **calibrated** with an isotonic regression on 2017, a year it has not seen (Zadrozny & Elkan 2002). Calibration matters because the firm scores add probabilities (expected breakthroughs): an uncalibrated forest gave the recent families a mean of 0.25–0.49 against a base rate of 0.107. We check the calibration honestly: a calibrator fitted on the validation-year predictions is tested on the test-year predictions (reliability table, expected calibration error, Brier score).
2. **Distribution shift**: the families we score (2022–2025) may not look like the training ones. For each feature, the population stability index (PSI) between training and scored families, and an adversarial validation (can a classifier tell them apart?). A large shift means the probabilities extrapolate: we say so next to the scores.
3. **The firms**: the **common set of Step 3, Section 11** (incumbents, entrants and the extension), so that every firm of the quadrant gets both scores. A firm with no family since 2022 gets a future score of 0 and the flag *no recent families* (it is not missing: it stopped filing).
4. **The future score** of a firm: percentile ranks inside the common set of (i) its mean calibrated probability, **shrunk towards the sector mean** with 10 pseudo-families (a firm with 5 families cannot top the ranking on luck), weight 0.5; (ii) its expected breakthroughs (sum of probabilities), weight 0.3; (iii) its momentum (share of its families filed since 2022), weight 0.2.

In [ ]:
BEST_FEATS = CLASSIC + OURS_UNIV
_val_row = bt_val[(bt_val["feature_set"] == PROPOSED) & (bt_val["model"] == BEST_MODEL_NAME)].iloc[0]
_test_row = bt_all[(bt_all["feature_set"] == PROPOSED) & (bt_all["model"] == BEST_MODEL_NAME)].iloc[0]
print(f"model: {BEST_MODEL_NAME} | PR-AUC validation {_val_row['PR_AUC']:.3f} | test {_test_row['PR_AUC']:.3f} "
      f"(Lift@10% {_test_row['Lift@10%']:.2f}x)")

# 1a. honest check of calibration: calibrator fitted on the validation-year predictions, tested on the test years
_pv = preds[(preds["feature_set"] == PROPOSED) & (preds["model"] == BEST_MODEL_NAME)]
_iso_chk = fit_isotonic_calibrator(_pv.loc[_pv["split"] == "validation", "y"], _pv.loc[_pv["split"] == "validation", "score"])
_pt = _pv[_pv["split"] == "test"]
rel_raw, ece_raw, brier_raw = reliability_table(_pt["y"], _pt["score"])
rel_cal, ece_cal, brier_cal = reliability_table(_pt["y"], _iso_chk.predict(_pt["score"]))
print(f"calibration on the test years: ECE raw {ece_raw:.3f} -> calibrated {ece_cal:.3f} | Brier raw {brier_raw:.3f} -> {brier_cal:.3f}")
display(rel_cal.round(3))

# 1b. the final model: trained before CALIBRATION_YEAR, calibrated on CALIBRATION_YEAR
final_train_mask = net["priority_year"].between(CALIBRATION_YEAR - LOOKBACK, CALIBRATION_YEAR - 1) & y_model.notna()
cal_mask = (net["priority_year"] == CALIBRATION_YEAR) & y_model.notna()
final_clf = make_classifiers(random_state=RANDOM_STATE)[BEST_MODEL_NAME]
final_clf.fit(signals.loc[final_train_mask, BEST_FEATS], y_model[final_train_mask].astype(int))
calibrator = fit_isotonic_calibrator(y_model[cal_mask].astype(int),
                                     final_clf.predict_proba(signals.loc[cal_mask, BEST_FEATS])[:, 1])

# 2. score the recent families: the sector and the extension of Step 3, Section 11
score_mask = (net["in_sector_v3"] | net["forced_extension"]) & net["priority_year"].between(*APPLY_YEARS)
recent_sec_ids = net.index[score_mask]
raw_probs = final_clf.predict_proba(signals.loc[recent_sec_ids, BEST_FEATS])[:, 1]
recent_probs = pd.Series(calibrator.predict(raw_probs), index=recent_sec_ids, name="pred_prob")
print(f"scored {len(recent_sec_ids):,} recent families ({int(net.loc[recent_sec_ids, 'forced_extension'].sum()):,} from the extension); "
      f"mean probability raw {raw_probs.mean():.3f} -> calibrated {recent_probs.mean():.3f} (training base rate {y_model[final_train_mask].mean():.3f}); "
      f"18-month window complete for {net.loc[recent_sec_ids, 'feature_window_complete'].mean():.0%} of them")

# 3. distribution shift between the training families and the scored families
_tr_X, _sc_X = signals.loc[final_train_mask, BEST_FEATS], signals.loc[recent_sec_ids, BEST_FEATS]
shift = pd.DataFrame({f: {"PSI": population_stability_index(_tr_X[f], _sc_X[f]),
                          "train median": _tr_X[f].median(), "scored median": _sc_X[f].median()} for f in BEST_FEATS}).T
shift["shift"] = pd.cut(shift["PSI"], [-np.inf, 0.1, 0.25, np.inf], labels=["stable", "moderate", "large"])
shift = shift.sort_values("PSI", ascending=False)
shift.to_csv(OUT_DIR / "distribution_shift.csv")
_rng = np.random.default_rng(RANDOM_STATE)
_a = _tr_X.iloc[_rng.permutation(len(_tr_X))[:5000]]
_b = _sc_X.iloc[_rng.permutation(len(_sc_X))[:5000]]
ADV_AUC = adversarial_validation_auc(_a, _b, random_state=RANDOM_STATE)
print(f"adversarial validation AUC (0.5 = no shift): {ADV_AUC:.2f}; features with a large shift (PSI > 0.25): "
      f"{', '.join(shift.index[shift['shift'] == 'large']) or 'none'}")
display(shift.round(3).head(12))

# Sector-enriched rule-based scorecard (diagnostic column)
_sec_rule_dirs = {k: d["direction"] for k, d in SIGNALS.items() if d["direction"] != 0 and k not in DYNAMIC_2Y}
sec_rule_scores = rule_based_future_score(signals.loc[recent_sec_ids], _sec_rule_dirs, year_series=net.loc[recent_sec_ids, "priority_year"])

# 4. families -> firms: the harmonised applicants of Step 0 (same MERGE as Step 3) + the extension of Step 3, Section 11
fam_app_step0 = load_step0("family_applicants_grouped")
MERGE = {
    "UNITREE": ["HANGZHOU YUSHU TECH"],
    "AGIBOT": ["SHANGHAI ZHIYUAN XINCHUANG TECHNOLOGY"],
    "XIAOMI ROBOT": ["BEIJING XIAOMI ROBOT TECH"],
    "HONDA RESEARCH INSTITUTE EUROPE": ["HONDA RESEARCH INST EUROPE", "HONDA RESEARCH INSTITUTE EUROPE"],
    "ALDEBARAN ROBOTICS": ["ALDEBARAN ROBOTICS", "软银机器人欧洲公司", "奥尔德巴伦机器人公司"],
    "PANASONIC": ["PANASONIC", "MATSUSHITA ELECT IND"],
}
INST_PAT = r"\b(?:UNIV\w*|INST\w*|ACADEM\w*|COLLEGE|SCHOOL|HOSPITAL|LAB\w*|CENT(?:ER|RE)|DAXUE|XUEYUAN)\b"
name_to_firm = {old: new for new, olds in MERGE.items() for old in olds}
fa = fam_app_step0.copy()
fa["firm"] = fa["applicant"].map(name_to_firm).fillna(fa["applicant"])
if "psn_sector" in fa.columns:
    _is_corp = fa["psn_sector"].astype(str).str.upper().str.contains("COMPANY")
else:
    _atype = fa["applicant_type"].astype(str).str.upper() if "applicant_type" in fa.columns else pd.Series("COMPANY", index=fa.index)
    _is_corp = _atype.isin(["COMPANY", "UNKNOWN"])
_is_inst = fa["firm"].astype(str).str.upper().str.contains(INST_PAT, regex=True) & ~fa["firm"].astype(str).str.upper().str.contains("HONDA")
fa["is_firm"] = _is_corp & ~_is_inst
fa_firms = fa[fa["is_firm"]][["docdb_family_id", "firm"]].drop_duplicates()

ext_firm_fams = pd.DataFrame(columns=["docdb_family_id", "firm"])
if QUAD_FAMS_FILE.exists():
    _qf = pd.read_csv(QUAD_FAMS_FILE)
    ext_firm_fams = _qf[_qf["source"] == "extension"][["docdb_family_id", "firm"]].drop_duplicates()
all_firm_families = pd.concat([fa_firms, ext_firm_fams], ignore_index=True).drop_duplicates()

recent_fam_scores = (all_firm_families[all_firm_families["docdb_family_id"].isin(recent_sec_ids)]
                     .assign(priority_year=lambda d: d["docdb_family_id"].map(net["priority_year"]),
                             prob=lambda d: d["docdb_family_id"].map(recent_probs),
                             pred_prob_raw=lambda d: d["docdb_family_id"].map(pd.Series(raw_probs, index=recent_sec_ids)),
                             rule_score=lambda d: d["docdb_family_id"].map(sec_rule_scores),
                             has_pct=lambda d: d["docdb_family_id"].map(signals["has_pct"]),
                             window_complete=lambda d: d["docdb_family_id"].map(net["feature_window_complete"]),
                             source=lambda d: np.where(d["docdb_family_id"].map(net["forced_extension"]), "extension", "sector")))
recent_fam_scores.to_csv(OUT_DIR / "recent_family_future_scores.csv", index=False)

# 5. the common set of firms (Step 3, Section 11), and the future score of each of them
step3_present = (pd.read_csv(STEP3_DIR / "present_score_quadrant.csv") if (STEP3_DIR / "present_score_quadrant.csv").exists()
                 else pd.DataFrame(columns=["firm", "score"]))
if len(step3_present):
    QUADRANT_FIRMS = step3_present["firm"].tolist()
else:
    _cnt = recent_fam_scores.groupby("firm")["docdb_family_id"].nunique()
    QUADRANT_FIRMS = sorted(_cnt[_cnt >= QCFG["min_recent_families"]].index)
    print("WARNING: outputs/step3/present_score_quadrant.csv not found: run Step 3 Section 11 first. "
          "Fallback: firms with recent families only, so the quadrant cannot be built.")
SECTOR_PRIOR = float(recent_probs[net.loc[recent_sec_ids, "in_sector_v3"].to_numpy()].mean())
firm_future = common_future_scores(recent_fam_scores[["docdb_family_id", "firm", "prob"]], QUADRANT_FIRMS, all_firm_families,
                                   prior=SECTOR_PRIOR, strength=SHRINK_K, weights=FUTURE_WEIGHTS)
_diag = recent_fam_scores.groupby("firm").agg(
    pct_share_pct=("has_pct", lambda s: s.mean() * 100),
    rule_based_mean=("rule_score", "mean"),
    window_complete_pct=("window_complete", lambda s: s.mean() * 100),
    extension_families=("source", lambda s: int((s == "extension").sum())),
)
firm_future = firm_future.join(_diag)
firm_future.reset_index().to_csv(OUT_DIR / "future_score.csv", index=False)
print(f"future score of the {len(firm_future)} firms of the common set (sector prior {SECTOR_PRIOR:.3f}, shrinkage {SHRINK_K} pseudo-families):")
display(firm_future.round(3).head(30))


def _plot_future_scores():
    top = firm_future.head(20).sort_values("future_score", ascending=True)
    cols = ["#7f8c8d" if nr else ("#8e44ad" if ex > 0 else "#e67e22")
            for nr, ex in zip(top["no recent families"], top["extension_families"].fillna(0))]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.2, 6.0))
    ax1.barh([f.title()[:28] for f in top.index], top["future_score"], color=cols)
    for k, (v, n) in enumerate(zip(top["future_score"], top["recent families"])):
        ax1.text(v + 1.0, k, f"{v:.0f} (n={int(n)})", va="center", fontsize=8)
    ax1.set_xlim(0, 118)
    ax1.set_xlabel("Future score (0–100, ranks inside the common set)")
    ax1.set_title("Future score (purple = uses the extension)")
    ax1.spines[["top", "right"]].set_visible(False)
    ax2.barh([f.title()[:28] for f in top.index], top["mean prob (shrunk)"], color=cols, alpha=0.85)
    ax2.axvline(SECTOR_PRIOR, color="black", ls="--", lw=1, label=f"sector mean {SECTOR_PRIOR:.3f}")
    ax2.set_yticks([])
    ax2.set_xlabel("Mean calibrated probability, shrunk towards the sector mean")
    ax2.set_title("Average portfolio quality")
    ax2.legend(fontsize=8, loc="lower right")
    ax2.spines[["top", "right"]].set_visible(False)
    plt.suptitle(f"Section 6 · Future score of the common set of firms ({APPLY_YEARS[0]}–{APPLY_YEARS[1]} families)", y=1.02, fontweight="bold")
    plt.tight_layout()
    plt.show()


_plot_future_scores()
del _plot_future_scores

> **⚠ Rerun needed (review of 28 Nov 2026).** The code above this reading changed; the numbers below are those of the previous PROD run and must be checked, and rewritten if they moved, after *Restart and Run All* on PROD.

**Our reading of the future scores.** *To rewrite*: the calibration check (ECE before and after), the distribution shift (adversarial AUC, the features with a large PSI), then the firms: the top of the ranking with the number of recent families behind each score, the firms with no recent families (score 0, they stopped filing), and the firms that depend on the extension.

### 6.2 The humanoid makers our title rule misses

In the first version these firms were scored separately, on the few of their families that happened to be in the sampled universe (4 for Boston Dynamics). Now their robotics families since 2015 are part of the common set (Step 3, Section 11), added to the universe and scored with everyone else. The table shows them apart, with how many of their recent families come from the extension; until the reading of the extension passes (Step 3, Section 11), their scores are flagged *unvalidated*.

In [ ]:
_ext_firms = firm_future[firm_future["extension_families"].fillna(0) > 0]
ext_players_scores = _ext_firms[["recent families", "extension_families", "mean prob (shrunk)", "expected breakthroughs",
                                 "momentum", "pct_share_pct", "future_score"]]
if len(ext_players_scores):
    ext_players_scores.to_csv(OUT_DIR / "missing_players_robotics_scores.csv")
    display(ext_players_scores.round(3))
else:
    print("no firm of the common set uses the extension (Step 3 Section 11 not run, or the extension was rejected)")

> **⚠ Rerun needed (review of 28 Nov 2026).** The code above this reading changed; the numbers below are those of the previous PROD run and must be checked, and rewritten if they moved, after *Restart and Run All* on PROD.

**Our reading of the humanoid makers our rule misses.** *To rewrite* from the table above, with the status of the extension reading (Step 3, Section 11).

### 6.3 Firm-level backtest: did the score of 2016 point to the firms that mattered next?

The family-level backtest of Section 4 says that the model ranks families. The deck ranks **firms**, so we test that too, as the kick-off asks (*freeze a date, compute the score with the data of that date, measure what happened afterwards*):

- **freeze** at the end of 2016. The model is trained only on labels known then (priority years 2008–2011, whose five years of citations are complete in 2016) and scores the sector families of 2013–2016, with the point-in-time features;
- each applicant of the sector with at least 3 families in 2013–2016 gets the future score of Section 6 (same shrinkage and weights), computed with the data of 2016;
- **what happened next**: (a) its families in the sector in 2017–2022 (did it keep building the field?) and (b) the share of its 2013–2016 families that turned out to be top-10% breakthroughs (known today);
- **baselines**: the firm's number of families in 2013–2016 (size) and its share of PCT families.

The units are few (a small sector before 2017), so we read the sign and the interval of Spearman's rho, not the decimals.

In [ ]:
FREEZE = 2016
WIN = (FREEZE - 3, FREEZE)
_train_f = net["priority_year"].between(YEAR_MIN, FREEZE - HORIZON) & y_model.notna()
clf_f = make_classifiers(random_state=RANDOM_STATE)[BEST_MODEL_NAME]
clf_f.fit(signals.loc[_train_f, BEST_FEATS], y_model[_train_f].astype(int))
_ids_f = net.index[net["in_sector_v3"] & net["priority_year"].between(*WIN)]
probs_f = pd.Series(clf_f.predict_proba(signals.loc[_ids_f, BEST_FEATS])[:, 1], index=_ids_f)

fa_all = fa[["docdb_family_id", "firm"]].drop_duplicates()            # every applicant type: more units for the test
fa_all = fa_all[fa_all["docdb_family_id"].isin(set(net.index[net["in_sector_v3"]]))]
fa_all = fa_all.assign(priority_year=lambda d: d["docdb_family_id"].map(net["priority_year"]))
win_rows = fa_all[fa_all["docdb_family_id"].isin(_ids_f)]
units = win_rows.groupby("firm")["docdb_family_id"].nunique()
units = units[units >= 3].index.tolist()
fut_f = common_future_scores(win_rows.assign(prob=lambda d: d["docdb_family_id"].map(probs_f))[["docdb_family_id", "firm", "prob"]],
                             units, fa_all[fa_all["priority_year"] <= FREEZE], prior=float(probs_f.mean()),
                             strength=SHRINK_K, weights=FUTURE_WEIGHTS)
later = fa_all[fa_all["priority_year"].between(FREEZE + 1, LAST_COMPLETE_YEAR)].groupby("firm")["docdb_family_id"].nunique()
bt_f = pd.DataFrame(index=pd.Index(units, name="firm"))
bt_f["future score (2016)"] = fut_f["future_score"]
bt_f["size 2013-2016"] = win_rows.groupby("firm")["docdb_family_id"].nunique()
bt_f["PCT share 2013-2016"] = win_rows.assign(p=lambda d: d["docdb_family_id"].map(signals["has_pct"])).groupby("firm")["p"].mean()
bt_f["families 2017-2022 (what happened)"] = later.reindex(units).fillna(0)
bt_f["breakthrough share of 2013-2016 families (what happened)"] = (
    win_rows.assign(v=lambda d: d["docdb_family_id"].map(net["valuable"])).groupby("firm")["v"].mean())
firm_bt = pd.DataFrame([
    {"score": s, "outcome": o, **spearman_with_ci(bt_f[s], bt_f[o], n_boot=1000, seed=RANDOM_STATE)}
    for s in ["future score (2016)", "size 2013-2016", "PCT share 2013-2016"]
    for o in ["families 2017-2022 (what happened)", "breakthrough share of 2013-2016 families (what happened)"]
])
firm_bt.to_csv(OUT_DIR / "firm_level_backtest.csv", index=False)
print(f"firm-level backtest: {len(units)} applicants with at least 3 sector families in {WIN[0]}-{WIN[1]}")
display(firm_bt.round(3))

> **⚠ Rerun needed (review of 28 Nov 2026).** The code above this reading changed; the numbers below are those of the previous PROD run and must be checked, and rewritten if they moved, after *Restart and Run All* on PROD.

**Our reading of the firm-level backtest.** *To write after the PROD run*: does the 2016 future score correlate with what the firms did next better than size and the PCT share? Report rho with its interval; if the interval of our score overlaps that of size, say that the firm score does not beat a count of families on this sample.

## 7. The data of the Magic Quadrant, sanity checks and deck answer

Both axes are now computed **for the same firms**, the common set of Step 3, Section 11:

- **y = present score as of today** (Step 3, Section 11, `present_score_quadrant.csv`), flagged *provisional* when it rests on 2 or 3 of the 4 indicators;
- **x = future score** (Section 6), 0 with the flag *no recent families* when the firm filed nothing since 2022;
- bubble = families of the firm (sector + extension), colour = Jaffe technological proximity to Electrolux (Step 3, Section 11);
- the cut lines are the **medians of the common set** on each axis (a convention: Step 5 may choose others and must say why);
- firms that depend on an unvalidated extension are drawn hollow.

The file `outputs/step4/magic_quadrant_data.csv` is the input of Step 5. The preview below is only a check that every firm can be placed.

In [ ]:
pq_path = STEP3_DIR / "present_score_quadrant.csv"
px_path = STEP3_DIR / "electrolux_proximity_quadrant.csv"
present_q = pd.read_csv(pq_path).set_index("firm") if pq_path.exists() else pd.DataFrame(columns=["score"])

mq = pd.DataFrame(index=pd.Index(QUADRANT_FIRMS, name="firm"))
mq["present_score"] = present_q["score"].reindex(mq.index)
for col in ("provisional", "indicators used", "families", "living families", "source", "extension status", "status"):
    if col in present_q.columns:
        mq[col] = present_q[col].reindex(mq.index)
mq["future_score"] = firm_future["future_score"].reindex(mq.index)
mq["no recent families"] = firm_future["no recent families"].reindex(mq.index)
mq["recent families"] = firm_future["recent families"].reindex(mq.index).fillna(0).astype(int)
mq["mean prob (shrunk)"] = firm_future["mean prob (shrunk)"].reindex(mq.index)
mq["pct_share_pct"] = firm_future["pct_share_pct"].reindex(mq.index)
if px_path.exists():
    mq["electrolux_proximity"] = pd.read_csv(px_path).set_index("firm")["jaffe_proximity"].reindex(mq.index)
PRESENT_CUT = float(mq["present_score"].median())
FUTURE_CUT = float(mq["future_score"].median())
mq["quadrant"] = [quadrant_label(p, f, PRESENT_CUT, FUTURE_CUT) for p, f in zip(mq["present_score"], mq["future_score"])]
mq["unvalidated"] = mq.get("extension status", pd.Series("", index=mq.index)).fillna("").eq("unvalidated")
mq = mq.sort_values(["quadrant", "future_score"], ascending=[True, False])
mq.reset_index().to_csv(OUT_DIR / "magic_quadrant_data.csv", index=False)
print(f"{len(mq)} firms; cut lines: present {PRESENT_CUT:.1f}, future {FUTURE_CUT:.1f} (medians of the common set)")
print("firms per quadrant:", mq["quadrant"].value_counts().to_dict())
display(mq.round(2))


def _plot_quadrant():
    df = mq[mq["quadrant"] != "not placed"].copy()
    _n = df["families"].fillna(df["recent families"]) if "families" in df.columns else df["recent families"]
    sizes = np.clip(_n.astype(float) * 6.0, 40, 700)
    cvals = df["electrolux_proximity"] if "electrolux_proximity" in df.columns else pd.Series(0.25, index=df.index)
    cvals = cvals.fillna(cvals.median() if cvals.notna().any() else 0.25)
    fig, ax = plt.subplots(figsize=(10.5, 7.2))
    ax.axvline(FUTURE_CUT, color="#7f8c8d", ls="--", lw=1.0)
    ax.axhline(PRESENT_CUT, color="#7f8c8d", ls="--", lw=1.0)
    solid = ~df["unvalidated"]
    sc = ax.scatter(df.loc[solid, "future_score"], df.loc[solid, "present_score"], s=sizes[solid], c=cvals[solid],
                    cmap="viridis", vmin=float(cvals.min()), vmax=float(cvals.max()), alpha=0.85, edgecolor="black", linewidth=0.8, zorder=3)
    ax.scatter(df.loc[~solid, "future_score"], df.loc[~solid, "present_score"], s=sizes[~solid], facecolors="none",
               edgecolors="#8e44ad", linewidth=1.6, zorder=3, label="uses the extension (not yet validated)")
    for firm, r in df.iterrows():
        lbl = firm.title().replace(" Robotics", "").replace(" Technology", "").replace(" Intelligent", "")[:22]
        lbl += "*" if str(r.get("provisional", "")) == "True" else ""
        ax.annotate(lbl, (r["future_score"], r["present_score"]), xytext=(6, 5), textcoords="offset points", fontsize=8)
    plt.colorbar(sc, ax=ax, pad=0.02).set_label("Jaffe technological proximity to Electrolux")
    ax.set_xlabel("Future score (Step 4, 0–100)")
    ax.set_ylabel("Present score as of today (Step 3, Section 11, RRF)")
    ax.set_title("Data check for Step 5 · every firm of the common set on both axes (* = provisional present score)", fontweight="bold")
    xl, yl = ax.get_xlim(), ax.get_ylim()
    for txt, x, y, ha in (("LEADERS", xl[1], yl[1], "right"), ("CHALLENGERS", xl[0], yl[1], "left"),
                          ("VISIONARIES", xl[1], yl[0], "right"), ("NICHE PLAYERS", xl[0], yl[0], "left")):
        ax.text(x, y, txt, ha=ha, va="top" if y == yl[1] else "bottom", fontsize=9, fontweight="bold", color="#555")
    if (~solid).any():
        ax.legend(loc="lower center", fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


_plot_quadrant()
del _plot_quadrant

In [ ]:
# --- Sanity checks & deck answer ---
def check(label, ok):
    print(("OK   - " if ok else "FAIL - ") + label)
    assert ok, f"Check failed: {label}"


check("no scored family was used to train or calibrate the final model",
      len(set(recent_sec_ids) & (set(net.index[final_train_mask]) | set(net.index[cal_mask]))) == 0)
check("no extension family is labelled (never trained or tested on)", y_model[net["forced_extension"]].isna().all())
check("the backtest ran on every validation and test year", set(preds["test_year"].unique()) == set(BACKTEST_YEARS))
check("the model was chosen on the validation years only", set(val_preds["test_year"].unique()) == set(VALIDATION_YEARS))
check("our t0 model beats the random base rate on the test years", float(_test_row["PR_AUC"]) > float(y_test_all.mean()))
check("every firm of the common set has a future score", mq["future_score"].notna().all())
check("outputs saved", (OUT_DIR / "future_score.csv").exists() and (OUT_DIR / "magic_quadrant_data.csv").exists())
if mq["present_score"].isna().any():
    print("NOTE - firms without a present score (fewer than the minimum indicators):", ", ".join(mq.index[mq["present_score"].isna()]))

_gain = paired_tab.iloc[0]
top_fut = firm_future[~firm_future["no recent families"]].head(5)
top_str = ", ".join(f"{f.title()[:22]} ({s:.0f}, n={int(n)})" for f, s, n in
                    zip(top_fut.index, top_fut["future_score"], top_fut["recent families"]))
answer_box(
    "Which humanoid-robot firms show the strongest future value?",
    f"Top future scores of the common set ({APPLY_YEARS[0]}-{APPLY_YEARS[1]} families): {top_str}. "
    f"Quadrant data: {len(mq)} firms on both axes, " + ", ".join(f"{k} {v}" for k, v in mq["quadrant"].value_counts().items()) + ".",
    f"Out-of-time test years {TEST_YEARS} (model {BEST_MODEL_NAME} chosen on {VALIDATION_YEARS}; base rate {y_test_all.mean():.1%}): "
    f"PR-AUC {_test_row['PR_AUC']:.3f}, Lift@10% {_test_row['Lift@10%']:.2f}x; gain over the classic benchmark "
    f"{_gain['diff']:+.3f} (paired 95% CI {_gain['ci_low']:+.3f} to {_gain['ci_high']:+.3f}). "
    f"Calibration error on the test years {ece_raw:.3f} -> {ece_cal:.3f}; adversarial AUC of the scored families {ADV_AUC:.2f}.",
)
write_step4_report_index()